In [13]:
import pandas as pd
import os
import re
from pathlib import Path
from collections import defaultdict
import json

class IterativeLotteryAnalyzer:
    def __init__(self, base_path):
        """
        base_path: Path to model/lottery_ticket directory
        """
        self.base_path = Path(base_path)
        self.prune_data = {}
    def get_indiv_concepts(self,formula) -> list:
        concepts = []
        concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
        for c in concps:
            try:
                end_idx = c.index(')')
            except:
                end_idx = len(c)
            concepts.append(c[:end_idx])
        return concepts

    def load_all_csvs(self):
        """Load all CSV files from the directory structure"""
        print("Loading CSV files...")
        
        for prune_dir in sorted(self.base_path.iterdir()):
      
            if not prune_dir.is_dir():
                continue
                
            try:
                match = str(prune_dir.name).split("%")
            except:
                continue
                
            prune_level = match[0]
            self.prune_data[prune_level] = {}
            for csv_file in prune_dir.glob('Cluster*'):
                cluster_match = re.search(r'Cluster(\d+)', csv_file.name)
                if not cluster_match:
                    continue
                    
                cluster = cluster_match.group(1)
                
                try:
                    df = pd.read_csv(csv_file)
                    
                    if 'unit' not in df.columns or 'best_name' not in df.columns:
                        print(f"Warning: {csv_file} missing required columns")
                        continue
                    
                    df = df.dropna(subset=['unit', 'best_name'])
                    
                    concepts =[]
                    for row in df['best_name']:
                        concepts.append(set(self.get_indiv_concepts(row)))
                  
                    unit_to_concept = dict(zip(df['unit'], concepts))
                    concept=set()
                    for c in concepts:
                        concept.update(c)
                    self.prune_data[prune_level][cluster] = {
                        'concepts': concept,
                        'unit_to_concept': unit_to_concept,
                        'total_units': len(unit_to_concept),
                        'df': df
                    }
                    
                    
                    print(f"  Loaded: {prune_level}% prune, Cluster {cluster} - {len(concepts)} concepts, {len(unit_to_concept)} units")
                    
                except Exception as e:
                    print(f"Error loading {csv_file}: {e}")
        
        print(f"\nLoaded {len(self.prune_data)} pruning levels\n")
        return self
    
    def analyze_iterative_changes(self):
        """Compare each pruning level to the previous one (iteration-to-iteration)"""
        prune_levels = sorted(self.prune_data.keys(), key=float)
        
        if len(prune_levels) < 2:
            print("Need at least 2 pruning levels for iterative analysis")
            return
        
        print(f"\n{'='*80}")
        print(f"ITERATIVE ANALYSIS: Changes Between Consecutive Pruning Levels")
        print(f"{'='*80}\n")
        
        results = []
        baselines={}
        for i in range(len(prune_levels) - 1):
            prev_level = prune_levels[i]
            curr_level = prune_levels[i + 1]
            
            prev_data = self.prune_data[prev_level]
            curr_data = self.prune_data[curr_level]
            
            print(f"\n{'='*80}")
            print(f"FROM {prev_level}% → TO {curr_level}% PRUNING")
            print(f"{'='*80}")
            
            # Per-cluster analysis
            cluster_analysis = {}
            
            for cluster in sorted(curr_data.keys(), key=int):
                if cluster not in prev_data:
                    continue
                
                prev_cluster = prev_data[cluster]
                curr_cluster = curr_data[cluster]
                
                prev_concepts = prev_cluster['concepts']
                curr_concepts = curr_cluster['concepts']
                
                preserved = curr_concepts & prev_concepts
                lost = prev_concepts - curr_concepts
                
                
                new_concepts = curr_concepts - prev_concepts
                if prev_level=='0.0':
                    num_relearned=0
                    percent_relearned=0
                    numprevlost=0
                    baselines[cluster]=prev_concepts
        
                else:
                    num_relearned=len(new_concepts & prev_lost)
                    percent_relearned=num_relearned/len(prev_lost)
                    numprevlost=len(prev_lost)
                    
                prev_lost=lost
                
                preservation_rate = (len(preserved) / len(prev_concepts) * 100) if prev_concepts else 0
                
                # Unit changes
                units_removed = prev_cluster['total_units'] - curr_cluster['total_units']
                units_removal_pct = (units_removed / prev_cluster['total_units'] * 100) if prev_cluster['total_units'] else 0
                
                '''cluster_analysis[cluster] = {
                    'preserved': len(preserved),
                    'lost': len(lost),
                    'new': len(new_concepts),
                    'preserved_list': sorted(preserved),
                    'lost_list': sorted(lost),
                    'new_list': sorted(new_concepts),
                    'prev_units': prev_cluster['total_units'],
                    'curr_units': curr_cluster['total_units'],
                    'units_removed': units_removed,
                    'units_removal_pct': units_removal_pct,
                    'preservation_rate': preservation_rate,
                    'num_relearned':num_relearned,
                    'percent_relearned':percent_relearned,
                    'prev_lost':numprevlost
                }
                
                print(f"\n  Cluster {cluster}:")
                print(f"    Units: {prev_cluster['total_units']} → {curr_cluster['total_units']} (removed {units_removed}, {units_removal_pct:.1f}%)")
                print(f"    Concepts: {len(prev_concepts)} → {len(curr_concepts)}")
                print(f"    Preserved: {len(preserved)} ({preservation_rate:.1f}%)")
                print(f"    Lost: {len(lost)}")
                print(f"    New: {len(new_concepts)}")
                print(f"    num_relearned: {num_relearned}")
                print(f"    percent_relearned: {percent_relearned}")
                print(f"    prev_lost: {numprevlost}")'''
                print(f"    percent preseved from og same clus: {len(curr_concepts&baselines[cluster])}/{len(curr_concepts)} = {len(curr_concepts&baselines[cluster]) / len(curr_concepts)}")
                
                '''if lost and len(lost) <= 10:
                    print(f"    Lost concepts: {', '.join(sorted(lost))}")
                elif lost:
                    print(f"    Lost concepts (sample): {', '.join(list(sorted(lost)))}")
                
                if new_concepts and len(new_concepts) <= 10:
                    print(f"    New concepts: {', '.join(sorted(new_concepts))}")
                elif new_concepts:
                    print(f"    New concepts (sample): {', '.join(list(sorted(new_concepts)))}...")'''
            
            # Global analysis
            all_prev_concepts = set()
            all_curr_concepts = set()
            for cluster_data in prev_data.values():
                all_prev_concepts.update(cluster_data['concepts'])
            for cluster_data in curr_data.values():
                all_curr_concepts.update(cluster_data['concepts'])
            
            global_preserved = all_curr_concepts & all_prev_concepts
            global_lost = all_prev_concepts - all_curr_concepts
            global_new = all_curr_concepts - all_prev_concepts
            
            global_preservation_rate = (len(global_preserved) / len(all_curr_concepts) * 100) if all_prev_concepts else 0
            
            if prev_level=='0.0':
                percent_relearned = 0
                num_relearned = 0
                num_lost = 0
                baseline=all_prev_concepts
            else:
                percent_relearned = len(global_new & prev_global_lost)/len(prev_global_lost)
                num_relearned = len(global_new & prev_global_lost)
                num_lost = len(prev_global_lost)
            print(f"\n  GLOBAL CHANGES:")
            '''print(f"    Total Concepts: {len(all_prev_concepts)} → {len(all_curr_concepts)}")
            print(f"    Preservation Rate from prev oter: {global_preservation_rate:.1f}%")
            print(f"    Lost in this step: {len(global_lost)}")
            print(f"    New in this step: {len(global_new)}")
            print(f"    percent_relearned in this step: {percent_relearned}")
            print(f"    num_relearned in this step: {num_relearned}")
            print(f"    num_lost in prev step: {num_lost}")'''
            print(f"    globally preservred from og: {len(baseline & all_curr_concepts)}/{len(all_curr_concepts)}= {len(baseline & all_curr_concepts) / len(all_curr_concepts)}")
            
            ''''if global_lost:
                print(f"\n    Globally Lost Concepts in this step:")
                print(sorted(global_lost))
                print(f"total {len(global_lost)}")
            
            if global_new:
                print(f"\n    Globally New Concepts in this step:")
                print(sorted(global_new))
                print(f"total {len(global_new)}")
            
            results.append({
                'from_level': prev_level,
                'to_level': curr_level,
                'clusters': cluster_analysis,
                'global': {
                    'prev_total': len(all_prev_concepts),
                    'curr_total': len(all_curr_concepts),
                    'preserved': len(global_preserved),
                    'lost': len(global_lost),
                    'new': len(global_new),
                    'preservation_rate': global_preservation_rate,
                    'lost_concepts': sorted(global_lost),
                    'new_concepts': sorted(global_new),
                    'num_relearned': num_relearned,
                }
            })'''
            prev_global_lost=global_lost
        
        return results
    
    def concept_survival_rate(self):
        """Track which concepts survive the longest and which die first"""
        prune_levels = sorted(self.prune_data.keys(), key=int)
        
        print(f"\n{'='*80}")
        print(f"CONCEPT SURVIVAL ANALYSIS")
        print(f"{'='*80}\n")
        
        # Get all concepts from baseline
        baseline = self.prune_data[prune_levels[0]]
        all_concepts = set()
        for cluster_data in baseline.values():
            all_concepts.update(cluster_data['concepts'])
        
        # Track when each concept is lost
        concept_survival = {}
        for concept in all_concepts:
            last_seen = None
            for level in prune_levels:
                # Check if concept exists in this level
                exists = False
                for cluster_data in self.prune_data[level].values():
                    if concept in cluster_data['concepts']:
                        exists = True
                        last_seen = level
                        break
                if not exists and last_seen is not None:
                    break
            concept_survival[concept] = last_seen if last_seen else prune_levels[0]
        
        # Group by survival level
        survival_groups = defaultdict(list)
        for concept, last_level in concept_survival.items():
            survival_groups[last_level].append(concept)
        
        print("Concepts lost at each pruning step:\n")
        for level in prune_levels[1:]:  # Skip baseline
            prev_level = str(int(level) - (int(prune_levels[1]) - int(prune_levels[0])))
            if prev_level in survival_groups:
                concepts_lost = [c for c in survival_groups[prev_level]]
                if concepts_lost:
                    print(f"  Lost between {prev_level}% → {level}%: {len(concepts_lost)} concepts")
                    if len(concepts_lost) <= 15:
                        for c in sorted(concepts_lost):
                            print(f"    - {c}")
                    else:
                        for c in sorted(concepts_lost)[:10]:
                            print(f"    - {c}")
                        print(f"    ... and {len(concepts_lost) - 10} more")
                    print()
        
        # Survivors
        final_level = prune_levels[-1]
        survivors = survival_groups[final_level]
        print(f"\n  Concepts surviving to {final_level}% pruning: {len(survivors)}")
        if len(survivors) <= 30:
            print(f"  Survivors:")
            for c in sorted(survivors):
                print(f"    ✓ {c}")
        else:
            print(f"  Top survivors (sample):")
            for c in sorted(survivors)[:20]:
                print(f"    ✓ {c}")
            print(f"    ... and {len(survivors) - 20} more")
        
        return survival_groups
    
    def cluster_transfer_analysis(self):
        """Analyze if concepts move between clusters during pruning"""
        prune_levels = sorted(self.prune_data.keys(), key=int)
        
        print(f"\n{'='*80}")
        print(f"CLUSTER TRANSFER ANALYSIS")
        print(f"{'='*80}\n")
        
        for i in range(len(prune_levels) - 1):
            prev_level = prune_levels[i]
            curr_level = prune_levels[i + 1]
            
            print(f"\nFrom {prev_level}% → {curr_level}%:")
            
            # Track where each concept was and where it is now
            prev_concept_clusters = defaultdict(set)
            curr_concept_clusters = defaultdict(set)
            
            for cluster, cluster_data in self.prune_data[prev_level].items():
                for concept in cluster_data['concepts']:
                    prev_concept_clusters[concept].add(cluster)
            
            for cluster, cluster_data in self.prune_data[curr_level].items():
                for concept in cluster_data['concepts']:
                    curr_concept_clusters[concept].add(cluster)
            
            # Find concepts that changed clusters
            transferred = []
            for concept in curr_concept_clusters:
                if concept in prev_concept_clusters:
                    if prev_concept_clusters[concept] != curr_concept_clusters[concept]:
                        transferred.append({
                            'concept': concept,
                            'from': sorted(prev_concept_clusters[concept]),
                            'to': sorted(curr_concept_clusters[concept])
                        })
            
            if transferred:
                print(f"  Concepts that moved clusters: {len(transferred)}")
                for item in transferred[:15]:  # Show first 15
                    print(f"    '{item['concept']}': clusters {item['from']} → {item['to']}")
                if len(transferred) > 15:
                    print(f"    ... and {len(transferred) - 15} more")
            else:
                print(f"  No concepts transferred between clusters")
    
    def save_results(self, results, output_file='iterative_analysis_resultsBowman.json'):
        """Save analysis results to JSON"""
        def convert_sets(obj):
            if isinstance(obj, set):
                return sorted(list(obj))
            elif isinstance(obj, dict):
                return {k: convert_sets(v) for k, v in obj.items()}
            elif isinstance(obj, list):
                return [convert_sets(item) for item in obj]
            return obj
        
        results_serializable = convert_sets(results)
        
        with open(output_file, 'w') as f:
            json.dump(results_serializable, f, indent=2)
        print(f"\n\nResults saved to {output_file}")


# Example usage
if __name__ == "__main__":
    base_path = "/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_7/Expls"  # Adjust this path
    
    analyzer = IterativeLotteryAnalyzer(base_path)
    analyzer.load_all_csvs()
    
    # Main iterative analysis (step-by-step changes)
    print("\n" + "="*80)
    print("RUNNING ITERATIVE ANALYSIS")
    print("="*80)
    results = analyzer.analyze_iterative_changes()
    analyzer.save_results(results)
    
    # Survival analysis (which concepts die first/last)
    analyzer.concept_survival_rate()
    
    # Transfer analysis (concepts moving between clusters)
    analyzer.cluster_transfer_analysis()
    
    print("\n\nAnalysis complete!")


"""
EXAMPLE OUTPUT:
================================================================================

Loading CSV files...
  Loaded: 0% prune, Cluster 1 - 95 concepts, 612 units
  Loaded: 0% prune, Cluster 2 - 142 concepts, 1843 units
  Loaded: 0% prune, Cluster 3 - 178 concepts, 2156 units
  Loaded: 25% prune, Cluster 1 - 88 concepts, 459 units
  Loaded: 25% prune, Cluster 2 - 137 concepts, 1382 units
  Loaded: 25% prune, Cluster 3 - 174 concepts, 1617 units

Loaded 6 pruning levels

================================================================================
ITERATIVE ANALYSIS: Changes Between Consecutive Pruning Levels
================================================================================


================================================================================
FROM 0% → TO 25% PRUNING
================================================================================

  Cluster 1:
    Units: 612 → 459 (removed 153, 25.0%)
    Concepts: 95 → 88
    Preserved: 88 (92.6%)
    Lost: 7
    New: 0
    Lost concepts: [in], [on], [the cat], [with], a dog, red car, the house

  Cluster 2:
    Units: 1843 → 1382 (removed 461, 25.0%)
    Concepts: 142 → 137
    Preserved: 135 (95.1%)
    Lost: 7
    New: 2
    Lost concepts: [after], blue sky, green tree, small box, under table, very cold, yellow bird
    New concepts: moving forward, quick motion

  Cluster 3:
    Units: 2156 → 1617 (removed 539, 25.0%)
    Concepts: 178 → 174
    Preserved: 171 (96.1%)
    Lost: 7
    New: 3
    Lost concepts: big house, fast car, hot coffee, old man, tall building, warm day, young child
    New concepts: combined action, multi-step, sequence pattern

  GLOBAL CHANGES:
    Total Concepts: 415 → 399
    Preservation Rate: 96.1%
    Lost in this step: 16
    New in this step: 5

    Globally Lost Concepts in this step:
      - [in]
      - [on]
      - [the cat]
      - [with]
      - a dog
      - big house
      - blue sky
      - fast car
      - green tree
      - hot coffee
      - old man
      - red car
      - small box
      - tall building
      - the house
      - under table

    Globally New Concepts in this step:
      + combined action
      + moving forward
      + multi-step
      + quick motion
      + sequence pattern


================================================================================
FROM 25% → TO 43% PRUNING
================================================================================

  Cluster 1:
    Units: 459 → 349 (removed 110, 24.0%)
    Concepts: 88 → 78
    Preserved: 76 (86.4%)
    Lost: 12
    New: 2
    Lost concepts: [a], [an], [is], [was], black, brown, dark, large, near, round, small, white

  Cluster 2:
    Units: 1382 → 1050 (removed 332, 24.0%)
    Concepts: 137 → 120
    Preserved: 116 (84.7%)
    Lost: 21
    New: 4

  Cluster 3:
    Units: 1617 → 1229 (removed 388, 24.0%)
    Concepts: 174 → 158
    Preserved: 152 (87.4%)
    Lost: 22
    New: 6

  GLOBAL CHANGES:
    Total Concepts: 399 → 356
    Preservation Rate: 89.2%
    Lost in this step: 43
    New in this step: 12


================================================================================
FROM 43% → TO 57% PRUNING
================================================================================

  Cluster 1:
    Units: 349 → 262 (removed 87, 24.9%)
    Concepts: 78 → 65
    Preserved: 63 (80.8%)
    Lost: 15
    New: 2

  Cluster 2:
    Units: 1050 → 788 (removed 262, 24.9%)
    Concepts: 120 → 102
    Preserved: 96 (80.0%)
    Lost: 24
    New: 6

  Cluster 3:
    Units: 1229 → 922 (removed 307, 25.0%)
    Concepts: 158 → 135
    Preserved: 128 (81.0%)
    Lost: 30
    New: 7

  GLOBAL CHANGES:
    Total Concepts: 356 → 302
    Preservation Rate: 84.8%
    Lost in this step: 54
    New in this step: 15


================================================================================
CONCEPT SURVIVAL ANALYSIS
================================================================================

Concepts lost at each pruning step:

  Lost between 0% → 25%: 16 concepts
    - [in]
    - [on]
    - [the cat]
    - [with]
    - a dog
    - big house
    - blue sky
    - fast car
    - green tree
    - hot coffee
    ... and 6 more

  Lost between 25% → 43%: 43 concepts
    - [a]
    - [an]
    - [is]
    - [was]
    - black
    - brown
    - dark
    - large
    - near
    - round
    ... and 33 more

  Lost between 43% → 57%: 54 concepts
    - [all]
    - [at]
    - [but]
    - [for]
    - [from]
    - [or]
    - beautiful
    - bright
    - clean
    - empty
    ... and 44 more

  Concepts surviving to 76% pruning: 244
  Top survivors (sample):
    ✓ action
    ✓ animal
    ✓ building
    ✓ city
    ✓ eat
    ✓ food
    ✓ house
    ✓ human
    ✓ move
    ✓ person
    ✓ place
    ✓ run
    ✓ sleep
    ✓ walk
    ✓ water
    ... and 229 more


================================================================================
CLUSTER TRANSFER ANALYSIS
================================================================================

From 0% → 25%:
  Concepts that moved clusters: 3
    'person': clusters ['2', '3'] → ['3']
    'running': clusters ['1', '3'] → ['3']
    'the dog': clusters ['2', '3'] → ['2']

From 25% → 43%:
  Concepts that moved clusters: 5
    'animal': clusters ['2', '3'] → ['3']
    'eat': clusters ['2', '3'] → ['2']
    'house': clusters ['1', '3'] → ['3']
    'person': clusters ['3'] → ['2', '3']
    'water': clusters ['2', '3'] → ['2']

From 43% → 57%:
  Concepts that moved clusters: 8
    'action': clusters ['2', '3'] → ['3']
    'animal': clusters ['3'] → ['2', '3']
    'building': clusters ['3'] → ['2', '3']
    'eat': clusters ['2'] → ['2', '3']
    'food': clusters ['2', '3'] → ['3']
    'human': clusters ['2', '3'] → ['3']
    'move': clusters ['3'] → ['2', '3']
    'place': clusters ['2', '3'] → ['3']


Results saved to iterative_analysis_results.json

Analysis complete!
"""

Loading CSV files...
  Loaded: 0.0% prune, Cluster 1 - 1034 concepts, 1024 units
  Loaded: 0.0% prune, Cluster 2 - 1024 concepts, 1024 units
  Loaded: 0.0% prune, Cluster 3 - 934 concepts, 934 units
  Loaded: 25.0% prune, Cluster 1 - 1024 concepts, 1024 units
  Loaded: 25.0% prune, Cluster 2 - 1024 concepts, 1024 units
  Loaded: 25.0% prune, Cluster 3 - 938 concepts, 938 units
  Loaded: 43.75% prune, Cluster 1 - 1024 concepts, 1024 units
  Loaded: 43.75% prune, Cluster 2 - 1024 concepts, 1024 units
  Loaded: 43.75% prune, Cluster 3 - 964 concepts, 964 units
  Loaded: 57.812% prune, Cluster 1 - 1024 concepts, 1024 units
  Loaded: 57.812% prune, Cluster 2 - 1024 concepts, 1024 units
  Loaded: 57.812% prune, Cluster 3 - 953 concepts, 953 units
  Loaded: 68.359% prune, Cluster 1 - 1024 concepts, 1024 units
  Loaded: 68.359% prune, Cluster 2 - 1024 concepts, 1024 units
  Loaded: 68.359% prune, Cluster 3 - 955 concepts, 955 units
  Loaded: 76.27% prune, Cluster 1 - 1024 concepts, 1024 units


ValueError: invalid literal for int() with base 10: '0.0'

In [24]:
from collections import defaultdict
class LTH_Wanda_NumConcept_Comparison:
    def __init__(self):
        self.lottery_ticket_concepts = defaultdict(list)
        self.wanda_concepts = defaultdict(list)
        
    def get_indiv_concepts(self,formula) -> set:
        concepts = []
        concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
        for c in concps:
            try:
                end_idx = c.index(')')
            except:
                end_idx = len(c)
            concepts.append(c[:end_idx])
        return set(concepts)
    
    def lth_wanda_comparison(self, sparsity):
        for cluster in range(1,4):
            print(f"Cluster {cluster}:\n\tNumber of LTH Concpets: {len(self.lottery_ticket_concepts[sparsity][f'Cluster{cluster}'])}\n\tNumber of wanda concepts: {len(self.wanda_concepts[sparsity][f'Cluster{cluster}'])}")
        
    def get_lth_concepts(self, model, method, filename):
        path = os.path.join('/workspace/CCE_NLI', model.upper(), 'exp', method, filename, 'Expls')
        for pi in sorted(os.listdir(path)):
            print(pi)
            method_concepts = {}
            for cluster in range(1,4):
                concepts=set()
                try:
                    csv = pd.read_csv(f"{os.path.join(path, pi, f'Cluster{cluster}IOUS1024N.csv')}")
                except:
                    print(f"pass")
                    continue
                for formula in csv['best_name']:
                    
                    concepts=concepts.union(self.get_indiv_concepts(formula))
                    
                method_concepts.update({f'Cluster{cluster}': concepts})
            if method == 'lottery_ticket':
                self.lottery_ticket_concepts[pi]=(method_concepts)
            elif method == 'wanda':
                self.wanda_concepts[pi]=(method_concepts)
        print(self.wanda_concepts['0.0%Pruned'])
compare = LTH_Wanda_NumConcept_Comparison()
MODEL='BOWMAN'
FILENAME='Run0.25_3'
compare.get_lth_concepts(MODEL, 'lottery_ticket', FILENAME)
compare.get_lth_concepts(MODEL, 'wanda', FILENAME)
compare.lth_wanda_comparison('25.0%Pruned')
        
        
                              

0.0%Pruned
25.0%Pruned
43.75%Pruned
57.812%Pruned
68.359%Pruned
pass
76.27%Pruned
82.202%Pruned
[]
25.0%Pruned
43.75%Pruned
57.812%Pruned
68.359%Pruned
76.27%Pruned
[]
Cluster 1:
	Number of LTH Concpets: 37
	Number of wanda concepts: 43
Cluster 2:
	Number of LTH Concpets: 30
	Number of wanda concepts: 34
Cluster 3:
	Number of LTH Concpets: 6
	Number of wanda concepts: 15
